# Exam Timetable Optimization Using Hill Climbing
**Problem Statement 4 - AI Group Assignment**

This notebook implements a complete Hill Climbing local search algorithm to optimize exam timetables by minimizing conflicts and uneven slot distribution.

## Problem Overview
- **Courses**: 6 (C1-C6)
- **Exam Slots**: 4 (Day 1 Morning, Day 1 Afternoon, Day 2 Morning, Day 2 Afternoon)
- **Conflicting Pairs**: 8 (courses with common students)
- **Objective**: Minimize total cost = (conflicts × 10) + (excess courses × 2)
- **Ideal Solution**: Cost = 0 (no conflicts, balanced distribution)

## How It Works
1. Start with an initial timetable
2. Generate all neighboring timetables (move one course to a different slot)
3. Find the neighbor with minimum cost
4. If neighbor is better, move to it and repeat
5. Stop when no better neighbor exists (local optimum)

In [252]:
import time
import copy
from typing import List, Set, Tuple, Dict
import pandas as pd

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

Imports successful


## Module 1: Input Processing
Reads and stores problem configuration.

In [253]:
class InputProcessor:
    # Reads and stores problem configuration from a text file.

    def __init__(self, filename):
        self.num_courses = 0
        self.num_slots = 0
        self.conflicts = set()
        self.initial_timetable = []

        self.read_from_file(filename)

    def read_from_file(self, filename):
        # Read the entire input file
        with open(filename, 'r') as f:
            text = f.read()

        # Remove empty lines and split into lines
        lines = [line.strip()for line in text.splitlines()if line.strip()]

        idx = 0

        # Read N and S
        self.num_courses, self.num_slots = map(int, lines[idx].split())
        idx += 1

        # Read K
        k = int(lines[idx])
        idx += 1

        # Read conflicting course pairs
        for _ in range(k):
            c1, c2 = map(int, lines[idx].split())
            self.conflicts.add(frozenset([c1, c2]))
            idx += 1

        # Read initial timetable
        self.initial_timetable = list(map(int, lines[idx].split()))

        # Basic validation
        if len(self.initial_timetable) != self.num_courses:
            raise ValueError(
                f"Expected {self.num_courses} timetable values, "
                f"but found {len(self.initial_timetable)}."
            )

        if not all(
            1 <= slot <= self.num_slots
            for slot in self.initial_timetable
        ):
            raise ValueError(
                "Initial timetable contains an invalid slot number."
            )

print("InputProcessor defined")


InputProcessor defined


## Module 2: State Representation + Module 3: Cost Function
Represents an exam timetable and calculates its cost.

**Cost Formula:**
- Conflict Penalty = (# conflicting pairs in same slot) × 10
- Distribution Penalty = (# excess courses beyond 2 per slot) × 2
- Total Cost = Conflict Penalty + Distribution Penalty

In [254]:
class TimetableState:
    # This class represents an examination timetable state.
    
    def __init__(self, timetable: List[int], num_slots: int, conflicts: Set):
        self.timetable = copy.copy(timetable)  # timetable[i] = slot for course i
        self.num_slots = num_slots
        self.conflicts = conflicts
        self.num_courses = len(timetable)
    
    def calculate_cost(self) -> Tuple[int, int, int]:
        
        ## Here we calculate the total cost of the timetable.
        ## And then return: (conflict_penalty, distribution_penalty, total_cost)
        
        # STEP 1: Calculate conflict penalty
        conflict_count = 0
        for conflict_pair in self.conflicts:
            c1, c2 = list(conflict_pair)
            # Convert to 0-indexed
            c1_idx, c2_idx = c1 - 1, c2 - 1
            
            # Check if both courses assigned to same slot
            if self.timetable[c1_idx] == self.timetable[c2_idx]:
                conflict_count += 1
        
        conflict_penalty = conflict_count * 10
        
        # STEP 2: Calculate distribution penalty
        # Count courses per slot
        slot_counts = [0] * (self.num_slots + 1)  # 1-indexed slots
        for slot in self.timetable:
            slot_counts[slot] += 1
        
        # Count excess courses (beyond 2 per slot)
        total_excess = 0
        for slot in range(1, self.num_slots + 1):
            excess = max(0, slot_counts[slot] - 2)
            total_excess += excess
        
        distribution_penalty = total_excess * 2
        
        # STEP 3: Total cost
        total_cost = conflict_penalty + distribution_penalty
        
        return conflict_penalty, distribution_penalty, total_cost
    
    def get_cost(self) -> int:
        # Get total cost (wrapper for convenience).
        _, _, total = self.calculate_cost()
        return total
    
    def get_detailed_cost(self) -> Dict:
        # Get cost with all components.
        conflict_pen, dist_pen, total = self.calculate_cost()
        return {
            'conflict_penalty': conflict_pen,
            'distribution_penalty': dist_pen,
            'total_cost': total,
            'conflicts': conflict_pen // 10,
            'excess_courses': dist_pen // 2
        }
    
    def __str__(self) -> str:
        # Display timetable in readable format.
        result = []
        for course in range(1, self.num_courses + 1):
            slot = self.timetable[course - 1]
            result.append(f"C{course} -> Slot {slot}")
        return "\n".join(result)
    
    def get_slot_distribution(self) -> Dict:
        # Get which courses are in which slots.
        distribution = {i: [] for i in range(1, self.num_slots + 1)}
        for course_idx, slot in enumerate(self.timetable, 1):
            distribution[slot].append(course_idx)
        return distribution

print("TimetableState defined")

TimetableState defined


## Module 4: Neighbor Generation
Generates neighboring states by moving one course to a different slot.

**Total neighbors**: N × (S - 1) = 6 × 3 = 18

In [255]:
class NeighborGenerator:
    # Generates neighboring timetable states.
    
    @staticmethod
    def generate_neighbors(state: TimetableState) -> List[TimetableState]:
        
        # Generate all neighboring states by moving one course to a different slot.
        # Total neighbors = num_courses × (num_slots - 1)
        
        neighbors = []
        
        # For each course
        for course_idx in range(state.num_courses):
            current_slot = state.timetable[course_idx]
            
            # Try assigning to each other slot
            for new_slot in range(1, state.num_slots + 1):
                if new_slot != current_slot:  # Only different slots
                    # Create new state
                    new_timetable = copy.copy(state.timetable)
                    new_timetable[course_idx] = new_slot
                    
                    neighbor = TimetableState(new_timetable, state.num_slots, state.conflicts)
                    neighbors.append(neighbor)
        
        return neighbors

print("NeighborGenerator defined")

NeighborGenerator defined


## Module 5: Hill Climbing Algorithm
Implements the iterative improvement strategy.

**Algorithm Steps:**
1. Calculate cost of current state
2. Generate all neighbors
3. Find neighbor with minimum cost
4. If neighbor is better (strictly lower cost), move to it
5. Repeat until no better neighbor exists

In [256]:
class HillClimbingOptimizer:
    # Implements Hill Climbing for timetable optimization.
    
    def __init__(self, initial_state: TimetableState):
        self.initial_state = initial_state
        self.current_state = copy.deepcopy(initial_state)
        
        self.iterations = 0
        self.neighbors_evaluated = 0
        self.iteration_history = []  # Track each iteration
    
    def optimize(self, verbose: bool = True) -> Dict:
        # Run Hill Climbing until local optimum is reached.
        
        start_time = time.time()
        current_cost = self.current_state.get_cost()
        
        if verbose:
            print(f"Initial Cost: {current_cost}")
            print("-" * 50)
        
        while True:
            self.iterations += 1
            
            # Generate all neighbors
            neighbors = NeighborGenerator.generate_neighbors(self.current_state)
            self.neighbors_evaluated += len(neighbors)
            
            # Calculate cost of all neighbors and find the best
            best_neighbor = None
            best_neighbor_cost = current_cost
            
            for neighbor in neighbors:
                neighbor_cost = neighbor.get_cost()
                
                # Keep track of best neighbor (strictly better)
                if neighbor_cost < best_neighbor_cost:
                    best_neighbor_cost = neighbor_cost
                    best_neighbor = neighbor
            
            # If no better neighbor found, we're at local optimum
            if best_neighbor is None:
                if verbose:
                    print(f"\nIteration {self.iterations}")
                    print("No improving neighbor found — terminating (Local Optimum).")
                break
            
            # Move to best neighbor
            self.current_state = best_neighbor
            current_cost = best_neighbor_cost
            
            # Track iteration
            cost_details = self.current_state.get_detailed_cost()
            self.iteration_history.append({
                'iteration': self.iterations,
                'timetable': copy.copy(self.current_state.timetable),
                'cost_details': cost_details
            })
            
            if verbose:
                print(f"Iteration {self.iterations}:")
                print(f"  Cost: {current_cost}")
                print(f"  Conflicts: {cost_details['conflicts']}, "
                      f"Excess Courses: {cost_details['excess_courses']}")
        
        end_time = time.time()
        execution_time = end_time - start_time
        
        return {
            'final_state': self.current_state,
            'iterations': self.iterations,
            'neighbors_evaluated': self.neighbors_evaluated,
            'execution_time': execution_time,
            'initial_cost': self.initial_state.get_cost(),
            'final_cost': self.current_state.get_cost(),
            'history': self.iteration_history
        }

print("HillClimbingOptimizer defined")

HillClimbingOptimizer defined


## Module 6: Result Display
### Result Display and Utilities

In [257]:
class ResultDisplay:
    # Formats and displays results.

    @staticmethod
    def display_timetable_result(
        result: Dict,
        initial_timetable: List[int],
        num_slots: int,
        experiment_name: str = ""
    ):
        # Display results in required format.

        print("=" * 50)
        print("Algorithm: Hill Climbing")
        print("Problem: Exam Timetable Optimization")

        if experiment_name:
            print(f"Experiment: {experiment_name}")

        print("=" * 50)

        # Initial state
        print("\nInitial Timetable:")

        # Use the actual number of slots
        initial_state = TimetableState(
            initial_timetable,
            num_slots,
            result['final_state'].conflicts
        )

        print(initial_state)

        initial_details = initial_state.get_detailed_cost()

        print(f"\nInitial Conflict Cost = "
              f"{initial_details['conflict_penalty']}")

        print(f"Initial Distribution Cost = "
              f"{initial_details['distribution_penalty']}")

        print(f"Initial Total Cost = "
              f"{initial_details['total_cost']}")

        # Iteration history
        print("\n" + "-" * 50)

        for iteration_info in result['history']:

            it = iteration_info['iteration']
            timetable = iteration_info['timetable']
            cost = iteration_info['cost_details']

            print(f"\nIteration {it}")

            # Display best neighbor timetable
            print("Best Neighbor Timetable:")

            for course, slot in enumerate(timetable, 1):
                print(f"C{course} → Slot {slot}")

            print(f"\nConflict Cost = "
                  f"{cost['conflict_penalty']}")

            print(f"Distribution Cost = "
                  f"{cost['distribution_penalty']}")

            print(f"Total Cost = "
                  f"{cost['total_cost']}")

        # Final state
        print("\n" + "=" * 50)

        print("\nFinal Timetable:")
        print(result['final_state'])

        final_details = result['final_state'].get_detailed_cost()

        print(f"\nFinal Conflict Cost = "
              f"{final_details['conflict_penalty']}")

        print(f"Final Distribution Cost = "
              f"{final_details['distribution_penalty']}")

        print(f"Final Total Cost = "
              f"{final_details['total_cost']}")

        print(f"\nIterations = {result['iterations']}")

        print(f"Total Neighboring States Evaluated = "
              f"{result['neighbors_evaluated']}")

        print(f"Execution Time = "
              f"{result['execution_time']:.8f} seconds")

        print("Termination Reason: Local Optimum")

        print("=" * 50)

    @staticmethod
    def create_comparison_dataframe(experiments: List[Dict]) -> pd.DataFrame:
        # Create comparison table for multiple experiments.

        data = []

        for i, exp in enumerate(experiments, 1):

            data.append({
                'Experiment': i,
                'Initial State': str(exp['initial_timetable']),
                'Initial Cost': exp['initial_cost'],
                'Final State': str(exp['final_timetable']),
                'Final Cost': exp['final_cost'],
                'Cost Reduction': (
                    exp['initial_cost'] - exp['final_cost']
                ),
                'Iterations': exp['iterations'],
                'Neighbors Evaluated': exp['neighbors_evaluated'],
                'Execution Time': (
                    f"{exp['execution_time']:.8f}"
                )
            })

        return pd.DataFrame(data)

print("ResultDisplay defined")


ResultDisplay defined


## Problem Definition
Define the exam scheduling problem parameters.

In [258]:
input_processor = InputProcessor("exam_input.txt")

num_courses = input_processor.num_courses
num_slots = input_processor.num_slots
conflicts = input_processor.conflicts

initial_timetables = [
    input_processor.initial_timetable,
    [2, 3, 1, 1, 4, 4],
    [4, 2, 3, 1, 2, 3]
]

print(f"Problem Configuration:")
print(f"  Number of Courses: {num_courses}")
print(f"  Number of Slots: {num_slots}")
print(f"  Number of Conflicting pairs: {len(conflicts)}")
print(f"  Number of Neighbors per iteration: {num_courses * (num_slots - 1)}")
print(f"\nReady to run experiments")

Problem Configuration:
  Number of Courses: 6
  Number of Slots: 4
  Number of Conflicting pairs: 8
  Number of Neighbors per iteration: 18

Ready to run experiments


## Example 1: Analyze a Single State
Before running Hill Climbing, let's analyze an initial state.

In [259]:
# Analyze the first initial timetable
initial = [1, 1, 2, 3, 4, 2]
state = TimetableState(initial, num_slots, conflicts)

print("Initial Timetable:")
print(state)

print("\n" + "="*50)
cost_info = state.get_detailed_cost()
print(f"Conflict Penalty: {cost_info['conflict_penalty']}")
print(f"  -> {cost_info['conflicts']} conflicting pairs in same slot")
print(f"Distribution Penalty: {cost_info['distribution_penalty']}")
print(f"  -> {cost_info['excess_courses']} excess courses across all slots")
print(f"Total Cost: {cost_info['total_cost']}")

print("\n" + "="*50)
print("\nSlot Distribution:")
distribution = state.get_slot_distribution()
for slot in range(1, num_slots + 1):
    courses = distribution[slot]
    print(f"  Slot {slot}: Courses {courses} ({len(courses)} courses)")

print("\n" + "="*50)
print("\nNeighbor Analysis:")
neighbors = NeighborGenerator.generate_neighbors(state)
print(f"Total neighbors generated: {len(neighbors)}")

# Show top 5 best neighbors
neighbor_costs = [(n, n.get_cost()) for n in neighbors]
neighbor_costs.sort(key=lambda x: x[1])

print("\nTop 5 Best Neighbors:")
for i, (neighbor, cost) in enumerate(neighbor_costs[:5], 1):
    print(f"\n  Neighbor {i} (Cost: {cost})")
    # Find which course was moved
    for j in range(len(initial)):
        if initial[j] != neighbor.timetable[j]:
            print(f"  Course {j+1}: Slot {initial[j]} -> Slot {neighbor.timetable[j]}")

Initial Timetable:
C1 -> Slot 1
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Conflict Penalty: 20
  -> 2 conflicting pairs in same slot
Distribution Penalty: 0
  -> 0 excess courses across all slots
Total Cost: 20


Slot Distribution:
  Slot 1: Courses [1, 2] (2 courses)
  Slot 2: Courses [3, 6] (2 courses)
  Slot 3: Courses [4] (1 courses)
  Slot 4: Courses [5] (1 courses)


Neighbor Analysis:
Total neighbors generated: 18

Top 5 Best Neighbors:

  Neighbor 1 (Cost: 10)
  Course 1: Slot 1 -> Slot 3

  Neighbor 2 (Cost: 10)
  Course 1: Slot 1 -> Slot 4

  Neighbor 3 (Cost: 10)
  Course 3: Slot 2 -> Slot 4

  Neighbor 4 (Cost: 10)
  Course 6: Slot 2 -> Slot 3

  Neighbor 5 (Cost: 12)
  Course 2: Slot 1 -> Slot 2


## Experiment 1: Run Hill Climbing
Run the first optimization experiment.

In [260]:
print("\n" + "#" * 60)
print("EXPERIMENT 1")
print("#" * 60 + "\n")

initial_timetable = initial_timetables[0]

initial_state = TimetableState(initial_timetable,num_slots,conflicts)
optimizer = HillClimbingOptimizer(initial_state)

result1 = optimizer.optimize(verbose=False)
ResultDisplay.display_timetable_result(result1,initial_timetable,num_slots,"Initial State 1")


############################################################
EXPERIMENT 1
############################################################

Algorithm: Hill Climbing
Problem: Exam Timetable Optimization
Experiment: Initial State 1

Initial Timetable:
C1 -> Slot 1
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

--------------------------------------------------

Iteration 1
Best Neighbor Timetable:
C1 → Slot 3
C2 → Slot 1
C3 → Slot 2
C4 → Slot 3
C5 → Slot 4
C6 → Slot 2

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

Iteration 2
Best Neighbor Timetable:
C1 → Slot 3
C2 → Slot 1
C3 → Slot 1
C4 → Slot 3
C5 → Slot 4
C6 → Slot 2

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0


Final Timetable:
C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Final Conflict Cost = 0
Final Distribution Cost = 0
Final Total Cost = 0

Iterations = 3
Total Neighboring St

## Experiment 2: Different Initial State
Run with a different initial configuration.

In [261]:
print("\n" + "#" * 60)
print("EXPERIMENT 2")
print("#" * 60 + "\n")

initial_timetable = initial_timetables[1]
initial_state = TimetableState(initial_timetable, num_slots, conflicts)

optimizer = HillClimbingOptimizer(initial_state)
result2 = optimizer.optimize(verbose=False)

ResultDisplay.display_timetable_result(result2, initial_timetable, num_slots,"Initial State 2")


############################################################
EXPERIMENT 2
############################################################

Algorithm: Hill Climbing
Problem: Exam Timetable Optimization
Experiment: Initial State 2

Initial Timetable:
C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 1
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 4

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

--------------------------------------------------

Iteration 1
Best Neighbor Timetable:
C1 → Slot 2
C2 → Slot 3
C3 → Slot 3
C4 → Slot 1
C5 → Slot 4
C6 → Slot 4

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

Iteration 2
Best Neighbor Timetable:
C1 → Slot 2
C2 → Slot 3
C3 → Slot 3
C4 → Slot 1
C5 → Slot 2
C6 → Slot 4

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0


Final Timetable:
C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 -> Slot 4

Final Conflict Cost = 0
Final Distribution Cost = 0
Final Total Cost = 0

Iterations = 3
Total Neighboring St

## Experiment 3: Third Initial State
Run with yet another different configuration.

In [262]:
print("\n" + "#" * 60)
print("EXPERIMENT 3")
print("#" * 60 + "\n")

initial_timetable = initial_timetables[2]
initial_state = TimetableState(initial_timetable, num_slots, conflicts)

optimizer = HillClimbingOptimizer(initial_state)
result3 = optimizer.optimize(verbose=False)

ResultDisplay.display_timetable_result(result3, initial_timetable, num_slots,"Initial State 3")


############################################################
EXPERIMENT 3
############################################################

Algorithm: Hill Climbing
Problem: Exam Timetable Optimization
Experiment: Initial State 3

Initial Timetable:
C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 -> Slot 3

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

--------------------------------------------------

Iteration 1
Best Neighbor Timetable:
C1 → Slot 4
C2 → Slot 2
C3 → Slot 3
C4 → Slot 1
C5 → Slot 4
C6 → Slot 3

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

Iteration 2
Best Neighbor Timetable:
C1 → Slot 4
C2 → Slot 2
C3 → Slot 2
C4 → Slot 1
C5 → Slot 4
C6 → Slot 3

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0


Final Timetable:
C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 2
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 3

Final Conflict Cost = 0
Final Distribution Cost = 0
Final Total Cost = 0

Iterations = 3
Total Neighboring St

## Experimental Comparison
Compare results across all three experiments.

In [263]:
# Prepare data for comparison
experiments_results = [
    {
        'experiment': 1,
        'initial_timetable': initial_timetables[0],
        'initial_cost': result1['initial_cost'],
        'final_timetable': result1['final_state'].timetable,
        'final_cost': result1['final_cost'],
        'iterations': result1['iterations'],
        'neighbors_evaluated': result1['neighbors_evaluated'],
        'execution_time': result1['execution_time']
    },
    {
        'experiment': 2,
        'initial_timetable': initial_timetables[1],
        'initial_cost': result2['initial_cost'],
        'final_timetable': result2['final_state'].timetable,
        'final_cost': result2['final_cost'],
        'iterations': result2['iterations'],
        'neighbors_evaluated': result2['neighbors_evaluated'],
        'execution_time': result2['execution_time']
    },
    {
        'experiment': 3,
        'initial_timetable': initial_timetables[2],
        'initial_cost': result3['initial_cost'],
        'final_timetable': result3['final_state'].timetable,
        'final_cost': result3['final_cost'],
        'iterations': result3['iterations'],
        'neighbors_evaluated': result3['neighbors_evaluated'],
        'execution_time': result3['execution_time']
    }
]

# Create and display comparison table
comparison_df = ResultDisplay.create_comparison_dataframe(experiments_results)
print("\n" + "=" * 100)
print("EXPERIMENTAL COMPARISON")
print("=" * 100 + "\n")
display(comparison_df)


EXPERIMENTAL COMPARISON



,Experiment,Initial State,Initial Cost,Final State,Final Cost,Cost Reduction,Iterations,Neighbors Evaluated,Execution Time
0,1,"[1, 1, 2, 3, 4, 2]",20,"[3, 1, 1, 3, 4, 2]",0,20,3,54,0.00075459
1,2,"[2, 3, 1, 1, 4, 4]",20,"[2, 3, 3, 1, 2, 4]",0,20,3,54,0.00198984
2,3,"[4, 2, 3, 1, 2, 3]",20,"[4, 2, 2, 1, 4, 3]",0,20,3,54,0.00095153


## Analysis of Results
Analyze the comparison results.

In [264]:
print("\nANALYSIS OF RESULTS")
print("=" * 100)

# Get final costs
costs = [exp['final_cost'] for exp in experiments_results]

# 1. Best final solution
best_idx = costs.index(min(costs))

print("\n1. Best Final Solution:")
print(f"   Experiment {best_idx + 1}")
print(f"   Initial State = {experiments_results[best_idx]['initial_timetable']}")
print(f"   Final State = {experiments_results[best_idx]['final_timetable']}")
print(f"   Final Cost = {experiments_results[best_idx]['final_cost']}")


# 2. Did all experiments reach the same final timetable?
all_same = all(
    exp['final_timetable'] == experiments_results[0]['final_timetable']
    for exp in experiments_results
)

print("\n2. Did all experiments reach the same final timetable?")
print(f"   {all_same}")

if all_same:
    print("   All three experiments converged to the same final timetable.")
else:
    print("   The experiments reached different final timetables.")


# 3. Did Hill Climbing get stuck in different local optima?
unique_final_costs = set(exp['final_cost'] for exp in experiments_results)

print("\n3. Did Hill Climbing get stuck in different local optima?")

if len(unique_final_costs) > 1:
    print("   Yes. Different initial states resulted in different final costs.")
else:
    print("   No. All experiments reached the same final cost.")

print("   Since the final cost is 0 in all experiments,")
print("   all tested initial states reached an optimal solution.")


# 4. How did the initial state affect the number of iterations?
print("\n4. How did the initial state affect the number of iterations?")

for exp in experiments_results:
    print(
        f"   Experiment {exp['experiment']}: "
        f"{exp['iterations']} iterations"
    )

iteration_values = [exp['iterations'] for exp in experiments_results]

if len(set(iteration_values)) == 1:
    print("   The initial state did not affect the number of iterations.")
    print(f"   All experiments required {iteration_values[0]} iterations.")
else:
    print("   The initial state affected the number of iterations.")


# 5. How did the initial state affect the final cost?
print("\n5. How did the initial state affect the final cost?")

for exp in experiments_results:
    print(
        f"   Experiment {exp['experiment']}: "
        f"{exp['initial_cost']} → {exp['final_cost']}"
    )

final_cost_values = [exp['final_cost'] for exp in experiments_results]

if len(set(final_cost_values)) == 1:
    print("   The initial state did not affect the final cost.")
    print(f"   All experiments reached final cost {final_cost_values[0]}.")
else:
    print("   The initial state affected the final cost.")


# Overall conclusion
print("\n" + "=" * 100)
print("OVERALL CONCLUSION")
print("=" * 100)

print("Hill Climbing successfully optimized all three initial timetables.")
print("All experiments reached a final cost of 0.")
print("Therefore, all tested initial states reached an optimal solution.")



ANALYSIS OF RESULTS

1. Best Final Solution:
   Experiment 1
   Initial State = [1, 1, 2, 3, 4, 2]
   Final State = [3, 1, 1, 3, 4, 2]
   Final Cost = 0

2. Did all experiments reach the same final timetable?
   False
   The experiments reached different final timetables.

3. Did Hill Climbing get stuck in different local optima?
   No. All experiments reached the same final cost.
   Since the final cost is 0 in all experiments,
   all tested initial states reached an optimal solution.

4. How did the initial state affect the number of iterations?
   Experiment 1: 3 iterations
   Experiment 2: 3 iterations
   Experiment 3: 3 iterations
   The initial state did not affect the number of iterations.
   All experiments required 3 iterations.

5. How did the initial state affect the final cost?
   Experiment 1: 20 → 0
   Experiment 2: 20 → 0
   Experiment 3: 20 → 0
   The initial state did not affect the final cost.
   All experiments reached final cost 0.

OVERALL CONCLUSION
Hill Climbing

## Verification: Final Solutions

In [265]:
print("\nFINAL TIMETABLES VERIFICATION")
print("=" * 80)

results = [result1, result2, result3]

for i, result in enumerate(results, 1):
    state = result['final_state']
    cost_info = state.get_detailed_cost()
    
    print(f"\nExperiment {i}:")
    print(f"  Final Timetable: {state.timetable}")
    print(f"  \n  Detailed Assignment:")
    print(f"  {state}")
    print(f"\n  Cost Breakdown:")
    print(f"    Conflicts: {cost_info['conflicts']} pairs × 10 = {cost_info['conflict_penalty']}")
    print(f"    Excess Courses: {cost_info['excess_courses']} courses × 2 = {cost_info['distribution_penalty']}")
    print(f"    Total Cost: {cost_info['total_cost']}")
    
    # Verify no conflicts
    if cost_info['total_cost'] == 0:
        print(f"    OPTIMAL SOLUTION FOUND")
    else:
        print(f"    Not optimal")


FINAL TIMETABLES VERIFICATION

Experiment 1:
  Final Timetable: [3, 1, 1, 3, 4, 2]
  
  Detailed Assignment:
  C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

  Cost Breakdown:
    Conflicts: 0 pairs × 10 = 0
    Excess Courses: 0 courses × 2 = 0
    Total Cost: 0
    OPTIMAL SOLUTION FOUND

Experiment 2:
  Final Timetable: [2, 3, 3, 1, 2, 4]
  
  Detailed Assignment:
  C1 -> Slot 2
C2 -> Slot 3
C3 -> Slot 3
C4 -> Slot 1
C5 -> Slot 2
C6 -> Slot 4

  Cost Breakdown:
    Conflicts: 0 pairs × 10 = 0
    Excess Courses: 0 courses × 2 = 0
    Total Cost: 0
    OPTIMAL SOLUTION FOUND

Experiment 3:
  Final Timetable: [4, 2, 2, 1, 4, 3]
  
  Detailed Assignment:
  C1 -> Slot 4
C2 -> Slot 2
C3 -> Slot 2
C4 -> Slot 1
C5 -> Slot 4
C6 -> Slot 3

  Cost Breakdown:
    Conflicts: 0 pairs × 10 = 0
    Excess Courses: 0 courses × 2 = 0
    Total Cost: 0
    OPTIMAL SOLUTION FOUND


## Summary
**Problem Solved Successfully** 

### What We Accomplished:
1. Implemented complete Hill Climbing algorithm with 5 modular classes
2. Ran 3 different experiments with different initial states
3. All experiments converged to optimal solution (cost = 0)
4. Tracked all required metrics: iterations, neighbors, time, cost
5. Output is generated in the required format specified in the assignment
6. Provided detailed cost calculation and verification

### Key Results:
- Initial costs in the tested experiments: 20
- Final costs in the tested experiments: 0
- Iterations in the tested experiments: 3
- Neighbors evaluated in the tested experiments: 54
- Execution time: ~0.0005 seconds average (very fast) [Execution time is subject to change for each rerun]

### Algorithm Characteristics:
- **Type**: Local search, greedy, best-improvement strategy
- **Termination**: When no better neighbor exists (local optimum)
- **Complexity**: O(I × N × (S-1) × (K+S)) where I=iterations
- **Optimality**: Found global optimum for this problem (lucky!), else we will be stuck at local optimum

## Interactive Mode
Create your own experiment through custom initial state.

In [266]:
print("INTERACTIVE EXPERIMENT BUILDER")
print("=" * 80)

st = input("Enter your custom timetable: ")
custom_timetable = [int(x) for x in st.split()]

print(f"\nRunning experiment with: {custom_timetable}")
print("=" * 80)

custom_state = TimetableState(custom_timetable,num_slots,conflicts)

custom_optimizer = HillClimbingOptimizer(custom_state)
custom_result = custom_optimizer.optimize(verbose=False)

ResultDisplay.display_timetable_result(custom_result,custom_timetable,num_slots,"Custom Experiment")


INTERACTIVE EXPERIMENT BUILDER

Running experiment with: [1, 2, 1, 1, 1, 2]
Algorithm: Hill Climbing
Problem: Exam Timetable Optimization
Experiment: Custom Experiment

Initial Timetable:
C1 -> Slot 1
C2 -> Slot 2
C3 -> Slot 1
C4 -> Slot 1
C5 -> Slot 1
C6 -> Slot 2

Initial Conflict Cost = 30
Initial Distribution Cost = 4
Initial Total Cost = 34

--------------------------------------------------

Iteration 1
Best Neighbor Timetable:
C1 → Slot 1
C2 → Slot 2
C3 → Slot 3
C4 → Slot 1
C5 → Slot 1
C6 → Slot 2

Conflict Cost = 10
Distribution Cost = 2
Total Cost = 12

Iteration 2
Best Neighbor Timetable:
C1 → Slot 1
C2 → Slot 2
C3 → Slot 3
C4 → Slot 4
C5 → Slot 1
C6 → Slot 2

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0


Final Timetable:
C1 -> Slot 1
C2 -> Slot 2
C3 -> Slot 3
C4 -> Slot 4
C5 -> Slot 1
C6 -> Slot 2

Final Conflict Cost = 0
Final Distribution Cost = 0
Final Total Cost = 0

Iterations = 3
Total Neighboring States Evaluated = 54
Execution Time = 0.00052428 seconds
Ter

## Conclusion

This notebook demonstrates a fully functional Hill Climbing algorithm for exam timetable optimization. The implementation includes:

**Modules Implemented:**
1. InputProcessor - Reads problem configuration
2. TimetableState - Represents states and calculates costs
3. NeighborGenerator - Creates neighboring solutions
4. HillClimbingOptimizer - Implements the algorithm
5. ResultDisplay - Formats and displays results

**Features:**
- Efficient cost calculation (O(K+S) per state)
- Complete neighborhood exploration (18 neighbors)
- Strict improvement criterion (no horizontal moves)
- Comprehensive metrics tracking
- Output in exact required format

**Results:**
- All 3 experiments converged to optimal solution (cost = 0)
- Consistent performance across different initial states
- Very fast execution (~0.0005s average)